# Figure 5 g–i — spectral PAINT of *S. aureus* with NR4A (Ximea)

**Makes up:** Figure 5 g (render + inset), h (wavelength render + inset), i (pixel 3 QE of two regions); SI: the per-pixel wavelength distribution (`fig:Saureus_photons`) and FRC (`fig:SI_FRC_Saureus`, 109 nm).

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/Ximea/SAureus/PLL_1in10_Condition1/50ms_70p_515_LP515_bp650_200_2/` — raw Ximea TIFFs
(12 files, ~45 GB) plus the acquisition's MicroManager metadata. The deposition holds raw
data only, so this notebook fits it first.

## FAST_MODE

`FAST_MODE = True` (default) downloads and fits only the first TIFF (~4 GB), enough to run the whole
pipeline end to end, with far fewer localisations than the published figure. `FAST_MODE = False`
downloads and fits the full acquisition (~45 GB; several hours of fitting; run in `tmux`/`screen`).
Hand-picked regions are in camera pixels of the full acquisition; in `FAST_MODE` they may be sparse.

---
## 0 — Imports

In [ ]:
import json
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import colors as mcolors
import seaborn as sns

# _biostudies_download.py lives beside this notebook (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve()))
import _biostudies_download as bd

from pyS3M import IOFunctions
from pyS3M import HelperFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SpectralFunctions
from pyS3M import SR_Functions
from pyS3M import PlottingBase
from pyS3M import render
from pyS3M import postprocess as _postprocess
from pyS3M import NileRedFunctions
from pyS3M.FRCFunctions import fire
from pyS3M.DriftCorrectionFunctions import Drift_Correction_Functions
from pyS3M.LinkingFunctions import link_localisations

IO = IOFunctions.IO_Functions()
H_F = HelperFunctions.Helper_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
SupRes_F = SR_Functions.SuperRes_Functions()   # Ximea defaults (69 nm pixels)
plotter = PlottingBase.PublicationPlotter(dark_background=False)

print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True   # True: first TIFF only (~4 GB, illustrative). False: all 12 TIFFs (~45 GB, the published data).

PIXEL_SIZE_NM = 69.0   # Ximea camera pixel

REPO_ROOT = Path.cwd().resolve().parents[1]   # notebooks/figures -> repo root
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("outputs") / "Figure_5g-i_SAureus"   # git-ignored (bare outputs/ rule)
FIG_DIR   = OUT_ROOT / ("fast" if FAST_MODE else "full")
# Point DATA_DIR at a drive with room for the download. Keep one folder per acquisition and
# per FAST_MODE setting: the fit reads every TIFF inside it.
DATA_DIR  = OUT_ROOT / ("data_fast" if FAST_MODE else "data_full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

gain_map   = IO.read_tiff(CALIB_DIR / "gain.tif")
offset_map = IO.read_tiff(CALIB_DIR / "offset.tif")
variance   = IO.read_tiff(CALIB_DIR / "variance.tif")
read_noise = IO.read_tiff(CALIB_DIR / "readnoise.tif")
rqe        = IO.read_tiff(CALIB_DIR / "rqe.tif")

R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

print(f"FAST_MODE  : {FAST_MODE}")
print(f"Data dir   : {DATA_DIR.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")

---
## 2 — Download the raw data from BioStudies (`S-BIAD3210`)

Downloads the first TIFF (`FAST_MODE`) or the whole series, plus the acquisition's MicroManager
`*_metadata.txt` in both cases: it holds the camera ROI the frames were cropped to, which the fit
needs to crop the calibration maps. Re-running is safe: complete files are skipped and interrupted
ones resume.

In [ ]:
REMOTE_FOLDER = "2026_Multicolour_Paper/Ximea/SAureus/PLL_1in10_Condition1/50ms_70p_515_LP515_bp650_200_2"

downloaded = bd.download_acquisition(REMOTE_FOLDER, DATA_DIR, first_file_only=FAST_MODE)

# Camera ROI of the acquisition (the fitted xc/yc are relative to its corner).
roi_x, roi_y, width, height = IO.metadata_reader_imageJ(downloaded["metadata"][0])
print(f"ROI: x={roi_x} y={roi_y}, {width} x {height} px")

---
## 3 — Spot detection and fitting

The paper's settings: pfa 1e-3, sigma 1.5, fraction_true 0.2, peak wavelength 0.65 um, 12 px ROI.
`fit_imaging_data` treats the TIFFs as one series and writes `Localisations.h5` into `DATA_DIR`.
This takes about 1 h in FAST_MODE and several hours in full. A record of what was fitted is saved beside it, so re-running the notebook
reuses the fit unless the files or settings change (or `REFIT = True`).

In [ ]:
FIT_PARAMS = dict(pfa=1e-3, sigma=1.5, fraction_true=0.2, peak_wavelength=0.65, ROI_size=12, NA=1.49)

# Quick single-frame check of the spot detection before the full fit.
fig, axs = SupRes_F.example_spots_singleframe(
    DATA_DIR, pfa=FIT_PARAMS["pfa"], sigma=FIT_PARAMS["sigma"],
    fraction_true=FIT_PARAMS["fraction_true"], frame_index=10,
    variance=variance, read_noise=read_noise, rqe=rqe, offset_map=offset_map,
    gain_map=gain_map, peak_wavelength=FIT_PARAMS["peak_wavelength"],
)
plt.show()

In [ ]:
REFIT = False

fit_file = DATA_DIR / "Localisations.h5"
fit_record_file = DATA_DIR / "Localisations_fit_record.json"
fit_record = {"images": [p.name for p in downloaded["images"]], **FIT_PARAMS}

previous = json.loads(fit_record_file.read_text()) if fit_record_file.exists() else None
if REFIT or not fit_file.exists() or previous != fit_record:
    SupRes_F.fit_imaging_data(
        DATA_DIR, smoothing_function, gain_map, offset_map, rqe, read_noise,
        variance=variance, image_type=".tif", **FIT_PARAMS,
    )
    fit_record_file.write_text(json.dumps(fit_record, indent=1))
else:
    print(f"Reusing {fit_file} (same files and settings)")

---
## 4 — Quality-control filters

The cuts used for the paper, copied from the analysis notebook (sizes in nm, converted to camera px).

In [ ]:
loc_data = pd.read_hdf(fit_file)
print(f"Raw localisations: {len(loc_data):,}")

loc_data = loc_data[loc_data["xc_err"] < 50 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["yc_err"] < 50 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["xc_err"] > 0]
loc_data = loc_data[loc_data["yc_err"] > 0]

loc_data = loc_data[loc_data["s_x_err"] < 50 / PIXEL_SIZE_NM]
loc_data = loc_data[loc_data["s_y_err"] < 50 / PIXEL_SIZE_NM]
loc_data = loc_data[(loc_data["s_x"] > 100 / PIXEL_SIZE_NM) & (loc_data["s_x"] < 175 / PIXEL_SIZE_NM)]
loc_data = loc_data[(loc_data["s_y"] > 100 / PIXEL_SIZE_NM) & (loc_data["s_y"] < 175 / PIXEL_SIZE_NM)]

loc_data = loc_data[loc_data["A_B_err"] < 0.01]
loc_data = loc_data[loc_data["A_G_err"] < 0.01]
loc_data = loc_data[loc_data["A_R_err"] < 0.01]

loc_data = loc_data[loc_data["A_B"] > 0.005]
loc_data = loc_data[loc_data["A_G"] > 0.05]
loc_data = loc_data[loc_data["A_R"] > 0.5]

loc_data = loc_data[loc_data["bg_B_err"] < 0.15]
loc_data = loc_data[loc_data["bg_G_err"] < 0.15]
loc_data = loc_data[loc_data["bg_R_err"] < 0.15]

loc_data = loc_data[~((loc_data["spot_snr"] < 2) & (loc_data["photons"] < 500)
                      & (loc_data["spot_matched_filter_response"] < 100))]
loc_data = loc_data[loc_data["spot_background_std"] > 10]
loc_data = loc_data[loc_data["chi_sqr"] < 1.5]
print(f"After QC: {len(loc_data):,}")

---
## 5 — Drift correction (AIM) and linking

Localisations of one molecule in consecutive frames (within the median localisation error,
up to 2 dark frames) are merged into one.

In [ ]:
drift_corrector = Drift_Correction_Functions()

info = [{
    "Width": width,
    "Height": height,
    "Frames": int(loc_data["frame"].max()),
    "Pixelsize": PIXEL_SIZE_NM,
}]

corrected_locs, drift_result = drift_corrector.undrift(
    locs=loc_data.to_records(index=False),
    info=info,
    method="aim",
    segmentation=50,
    intersect_d=20 / PIXEL_SIZE_NM,
    roi_r=60 / PIXEL_SIZE_NM,
)
corrected_locs = pd.DataFrame(corrected_locs)

plt.plot(drift_result.drift_x * PIXEL_SIZE_NM, label="x")
plt.plot(drift_result.drift_y * PIXEL_SIZE_NM, label="y")
plt.xlabel("frame"); plt.ylabel("drift / nm"); plt.legend()
plt.show()

# Frame count from the fitted data, so FAST_MODE (first file only) gets the right number.
n_frames = int(loc_data["frame"].max()) + 1
link_r = 0.5 * (np.median(corrected_locs["xc_err"]) + np.median(corrected_locs["yc_err"]))
linked = link_localisations(corrected_locs, n_frames=n_frames, r_max=link_r, max_dark_time=2)
print(f"Linked localisations: {len(linked):,}")

In [ ]:
IO.write_h5_database(df=linked, filepath=DATA_DIR / "Localisations_Undrifted_Linked.h5", normalise_photons=False)

---
## 6 — Figure 5g — super-resolved render and inset

`CUT_SR` drops the empty lower-left corner of the field (in super-resolution pixels);
`ZOOM` is the inset (camera px, relative to the cut).

In [ ]:
OVERSAMPLING = 8
MIN_BLUR_WIDTH = 4 * PIXEL_SIZE_NM / 1000
CUT_SR = 500                              # SR px removed from the bottom and left
ZOOM = dict(x0=260, y0=340, size=50)      # camera px within the cut

info = [{"Width": width, "Height": height, "Frames": int(linked["frame"].max()), "Pixelsize": PIXEL_SIZE_NM}]
image = render.render(locs=linked.to_records(index=False), info=info, blur_method="smooth",
                      oversampling=OVERSAMPLING, min_blur_width=MIN_BLUR_WIDTH)[1]
cut = image[CUT_SR:, CUT_SR:]


def zoom_box(color="white"):
    return patches.Rectangle((ZOOM["x0"] * OVERSAMPLING, ZOOM["y0"] * OVERSAMPLING),
                             ZOOM["size"] * OVERSAMPLING, ZOOM["size"] * OVERSAMPLING,
                             linewidth=0.75, edgecolor=color, facecolor="none")


def zoom_crop(img):
    y0, x0, s = ZOOM["y0"] * OVERSAMPLING, ZOOM["x0"] * OVERSAMPLING, ZOOM["size"] * OVERSAMPLING
    return img[y0:y0 + s, x0:x0 + s]


fig, ax = plotter.two_column_plot(height=2, width=2)
ax, _ = plotter.image_plot(ax, cut, cmap="hot", vmax=np.percentile(image, 99.9),
                           pixelsize=PIXEL_SIZE_NM / OVERSAMPLING, scalebar=True)
ax.add_patch(zoom_box())
plt.savefig(FIG_DIR / "Image_SAureus.svg", dpi=600, format="svg")
plt.show()

fig, ax = plotter.one_column_plot()
ax, _ = plotter.image_plot(ax, zoom_crop(cut), cmap="hot", vmin=np.percentile(cut, 1),
                           vmax=np.percentile(cut, 99.9), pixelsize=PIXEL_SIZE_NM / OVERSAMPLING,
                           scalebar=True, scalebarsize=1000, scalebarlabel=r"1 $\mu$m")
plt.savefig(FIG_DIR / "SAureus_Zoomin.svg", dpi=600, format="svg")
plt.show()

---
## 7 — Segment the bacteria and fit a wavelength per 50 nm pixel

Bacteria are found by Li-thresholding the rendered image (much cheaper than DBSCAN at this
density); the Nile Red forward model then fits an emission peak to each 50 nm analysis pixel.

In [ ]:
seg_locs, seg_stats = _postprocess.segment_locs_by_rendered_image(
    linked,
    width=width,
    height=height,
    oversampling=8,
    pixel_size_nm=PIXEL_SIZE_NM,
    min_area_nm2=100.0,
    min_localisations=20,
    threshold_method="li",
    callback="console",
    blur_method="gaussian",
    verbose=True,
)
seg_locs = seg_locs.rename(columns={"aggregate_id": "cluster_id", "aggregate_area_nm2": "cluster_area_nm2"})
seg_stats = seg_stats.rename(columns={"aggregate_id": "cluster_id"})

clustered_file = DATA_DIR / "Localisations_Undrifted_Linked_clusteredlocs.h5"
IO.write_h5_database(df=seg_locs, filepath=clustered_file, append=False, normalise_photons=False)
IO.write_h5_database(df=seg_stats, filepath=DATA_DIR / "Localisations_Undrifted_Linked_averagedoverclusters.h5",
                     append=False, normalise_photons=False)
print(f"{seg_locs['cluster_id'].nunique()} bacteria, {len(seg_locs):,} localisations")

In [ ]:
nrf = NileRedFunctions.NileRed_Functions()
filters = [
    "semrock-ff01-650-200",
    "semrock-di03-r514-t1-25x36",
    "semrock-ff01-515-lp",
]
camera_params = {"pixel_QYs": pixel_QYs, "wavelength": wavelength}

# Fits a wavelength per 50 nm analysis pixel (at least 10 localisations each) and writes
# it back into the clustered-localisations file as wl_pixel / pixel_ix.
df_with_wavelengths, grid_info = nrf.fit_wavelengths_pixelated(
    h5_path=str(clustered_file),
    filter_names=filters,
    camera_parameters=camera_params,
    output_path=str(clustered_file),
    min_localisations=10,
    pixel_size_nm=50,
    aggregate_id_column="cluster_id",
)
print(f"Wavelength-fitted localisations: {len(df_with_wavelengths):,}")

---
## 8 — Figure 5h — wavelength render and inset

Hue = fitted wavelength (5th–95th percentile), brightness = localisation density.

In [ ]:
wl_min = np.percentile(df_with_wavelengths["wl_pixel"], 5)
wl_max = np.percentile(df_with_wavelengths["wl_pixel"], 95)

_, image_grey, image_colour = render.render(
    locs=df_with_wavelengths.to_records(index=False),
    oversampling=OVERSAMPLING,
    viewport=((0, 0), (np.max(df_with_wavelengths["yc"]), np.max(df_with_wavelengths["xc"]))),
    blur_method="gaussian_colour",
    cparam="wl_pixel",
    c_min=wl_min,
    c_max=wl_max,
)

# Replace the HSV brightness with the (percentile-clipped) density image.
vmax = np.percentile(image_grey, 99.5)
vmin = np.percentile(image_grey, 1)
hsv = mcolors.rgb_to_hsv(image_colour)
hsv[..., 2] = np.clip((image_grey - vmin) / (vmax - vmin), 0, 1)
image_adjusted = mcolors.hsv_to_rgb(hsv)
cut_colour = image_adjusted[CUT_SR:, CUT_SR:, :]

fig, ax = plotter.two_column_plot(height=2, width=2.25)
ax = plotter.colour_image_plot(ax, cut_colour, cbar="on", cbarlabel=r"$\lambda_{model}$/nm", cmap="jet",
                               c_min=wl_min, c_max=wl_max, pixelsize=PIXEL_SIZE_NM / OVERSAMPLING)
ax.add_patch(zoom_box())
plt.savefig(FIG_DIR / "Image_SAureus_Wavelength.svg", dpi=600, format="svg")
plt.show()

---
## 9 — Figure 5i — pixel 3 QE in two neighbouring regions

Regions A and B were picked by hand on the published image, in camera px of the fitted
coordinates (the same raw data, so they still apply after re-fitting).

In [ ]:
RECTS = [
    dict(x0=351, y0=425, w=4, h=5, color="#4363d8", label="Region A"),
    dict(x0=355, y0=424, w=4, h=5, color="#e6194b", label="Region B"),
]


def get_region_localisations(df, rectangles):
    """Localisations inside each rectangle (camera px)."""
    return [df[(df["xc"] >= r["x0"]) & (df["xc"] < r["x0"] + r["w"])
               & (df["yc"] >= r["y0"]) & (df["yc"] < r["y0"] + r["h"])].copy() for r in rectangles]


regions = get_region_localisations(df_with_wavelengths, RECTS)
for r, reg in zip(RECTS, regions):
    print(f"{r['label']}: {len(reg)} localisations")

# Inset of the wavelength render with the two regions, and each region on its own.
zoom_colour = zoom_crop(cut_colour)
zoom_x0_cam = CUT_SR / OVERSAMPLING + ZOOM["x0"]
zoom_y0_cam = CUT_SR / OVERSAMPLING + ZOOM["y0"]

fig, ax = plotter.one_column_plot()
ax = plotter.colour_image_plot(ax, zoom_colour, cbar="off", pixelsize=PIXEL_SIZE_NM / OVERSAMPLING,
                               scalebarsize=1000, scalebarlabel=r"1 $\mu$m")
region_images = []
for r in RECTS:
    x = (r["x0"] - zoom_x0_cam) * OVERSAMPLING
    y = (r["y0"] - zoom_y0_cam) * OVERSAMPLING
    w, h = r["w"] * OVERSAMPLING, r["h"] * OVERSAMPLING
    ax.add_patch(patches.Rectangle((x, y), w, h, linewidth=1, edgecolor=r["color"], facecolor="none"))
    region_images.append(zoom_colour[int(y):int(y + h), int(x):int(x + w)])
plt.savefig(FIG_DIR / "SAureus_Zoomin_Colour_Rects.svg", dpi=600, format="svg")
plt.show()

for i, img in enumerate(region_images, 1):
    fig, ax = plotter.one_column_plot()
    ax = plotter.colour_image_plot(ax, img, cbar="off", pixelsize=PIXEL_SIZE_NM / OVERSAMPLING,
                                   scalebarsize=100, scalebarlabel="100 nm")
    plt.savefig(FIG_DIR / f"SAureus_Zoomed_Area_{i}.svg", dpi=600, format="svg")
    plt.show()

In [ ]:
fig, ax = plotter.one_column_plot(width=2, height=1.5)
for r, reg in zip(RECTS, regions):
    if len(reg) < 2:
        print(f"{r['label']}: too few localisations to plot (expected in FAST_MODE)")
        continue
    ax = plotter.histogram_plot(ax, reg["A_R"], np.histogram_bin_edges(reg["A_R"], "fd"), density=False,
                                color=r["color"], alpha=0.5, xlabel="pixel 3 QE",
                                label=f"{r['label']}, n = {len(reg)}")
    ax.axvline(np.mean(reg["A_R"]), color=r["color"], ls="--", lw=1)
ax.set_xlim([0.5, 0.9])
ax.legend(fontsize=6)
plt.savefig(FIG_DIR / "SAureus_Histogram.svg", dpi=600, format="svg")
plt.show()

---
## 10 — SI — per-pixel wavelength distribution

Published: mean ~619 nm, standard deviation ~1 nm, no correlation with photons per pixel.

In [ ]:
averaged = df_with_wavelengths.groupby("pixel_ix", as_index=False).agg(
    xc=("xc", "mean"),
    yc=("yc", "mean"),
    photons=("photons", "sum"),
    wl_pixel=("wl_pixel", "mean"),
    n_locs=("frame", "count"),
)
averaged["log_photons"] = np.log10(averaged["photons"])
mean, std = np.mean(averaged["wl_pixel"]), np.std(averaged["wl_pixel"])

fig, axs = plotter.one_column_plot(width=1.5, height=2, npanels=2, ratios=[1, 1])
axs[0] = plotter.histogram_plot(axs[0], averaged["wl_pixel"],
                                bins=np.histogram_bin_edges(averaged["wl_pixel"], "fd"), xlabel="", density=True)
axs[0].axvline(mean, ls="--", color="red", label=r"$\mu$")
axs[0].axvspan(mean - std, mean + std, alpha=0.15, color="red", label=r"$\sigma$")
axs[0].set_ylabel(r"probability $\rho$")
axs[0].set_xticklabels([])
axs[0].legend(fontsize=6)

sns.kdeplot(data=averaged, x="wl_pixel", y="log_photons", ax=axs[1], fill=True)
axs[1].grid(True, lw=0.5, alpha=0.5, ls="--", color="grey")
axs[1].set_xlabel("mean wavelength/nm")
axs[1].set_ylabel("log(photons)")
for a in axs:
    a.set_xlim([615, 625])
plt.savefig(FIG_DIR / "SAureus_Analysis_Wavelength.svg", dpi=600, format="svg")
plt.show()

print(f"Per-pixel wavelength: {mean:.1f} +/- {std:.1f} nm over {len(averaged):,} pixels")

---
## 11 — SI — FRC resolution

Published: 109 +/- 1 nm.

In [ ]:
def frc_curve(locs):
    """FIRE resolution and FRC curve of one set of localisations (camera px, with frame).

    Passing the frame column makes fire() split the two halves by time blocks, not row order.
    """
    xyf = locs[["xc", "yc", "frame"]].to_numpy(dtype=float)
    xyf[:, 0] -= xyf[:, 0].min()
    xyf[:, 1] -= xyf[:, 1].min()
    nx = int(xyf[:, 0].max()) + 1
    ny = int(xyf[:, 1].max()) + 1
    res, frc, res_hi, res_lo = fire(positions=xyf, nx=nx, ny=ny, zoom=FRC_ZOOM, n_blocks=50,
                                    pixel_size_nm=PIXEL_SIZE_NM)
    # Spatial frequency of each FRC bin in cycles/nm, kept below the SR-pixel Nyquist limit.
    q = np.arange(len(frc)) / (max(nx, ny) * FRC_ZOOM)   # cycles per SR pixel
    keep = (q > 0) & (q < 0.6)
    q_nm = q / (PIXEL_SIZE_NM / FRC_ZOOM)
    return dict(res=res, res_hi=res_hi, res_lo=res_lo, q_nm=q_nm[keep], frc=frc[keep])


def plot_frc(ax, curve, label, color, span_color):
    ax = plotter.line_plot(ax, curve["q_nm"], curve["frc"], label=label, color=color)
    ax.axvline(1 / curve["res"], color=span_color, ls="-", lw=1, label=f"FIRE = {curve['res']:.0f} nm")
    ax.axvspan(1 / curve["res_lo"], 1 / curve["res_hi"], alpha=0.15, color=span_color)
    return ax


FRC_ZOOM = 8   # SR pixels per camera pixel

In [ ]:
frc_saureus = frc_curve(linked)
print(f"FIRE resolution: {frc_saureus['res']:.1f} nm "
      f"(+/- {abs(frc_saureus['res_hi'] - frc_saureus['res_lo']) / 2:.1f} nm)")

fig, ax = plotter.two_column_plot()
ax = plot_frc(ax, frc_saureus, "FRC (mean), NR4A", "darkred", "crimson")
ax.axhline(1 / 7, color="gray", ls="--", lw=1, label="1/7 threshold")
ax.set_xlabel("1 / spatial frequency (nm)", fontsize=8)
ax.set_ylabel("FRC", fontsize=8)
ax.set_ylim([0, 1.05])
ax.set_xscale("log")
tick_nm = [1000, 500, 200, 100, 50, 30, 20]
ax.set_xticks([1 / d for d in tick_nm])
ax.set_xticklabels([f"{d} nm" for d in tick_nm])
ax.set_xlim(1 / 1000, 1 / 15)
ax.legend(fontsize=7)
plt.savefig(FIG_DIR / "FRC_SAureus.svg", dpi=600, format="svg")
plt.show()